# DSA 4060 – Personalized Movie Recommender

**Name:** Justice Chawanda  
**Student ID:** 670444  
**Assigned User ID:** 5

**How the assigned user was determined.** The last two numeric digits of the Student ID `670444` are `44`,
and `44 MOD 40 = 4`, so the assigned User ID is `4 + 1 = `**`5`**.

All profiling, similarity scoring and recommendations in this notebook are computed for **User 5 only**.

---
## Task 1 – Load and Inspect the Data (3 marks)

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 80)

# The one parameter that personalizes this notebook.
STUDENT_ID = 670444
USER_ID = (STUDENT_ID % 100) % 40 + 1
LIKE_THRESHOLD = 4.0
N_RECOMMENDATIONS = 5

movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

print(f"Student ID {STUDENT_ID} -> last two digits {STUDENT_ID % 100} "
      f"-> {STUDENT_ID % 100} MOD 40 = {(STUDENT_ID % 100) % 40} -> Assigned User ID = {USER_ID}")

Student ID 670444 -> last two digits 44 -> 44 MOD 40 = 4 -> Assigned User ID = 5


### First five rows of each dataset (1 mark)

In [2]:
print("movies.csv - first five rows")
display(movies.head(5))

print("ratings.csv - first five rows")
display(ratings.head(5))

movies.csv - first five rows


,movie_id,title,genres
0,1,Inception,Sci-Fi|Thriller
1,2,Interstellar,Sci-Fi|Drama
2,3,The Matrix,Sci-Fi|Action
3,4,Arrival,Sci-Fi|Drama
4,5,Edge of Tomorrow,Sci-Fi|Action


ratings.csv - first five rows


,user_id,movie_id,rating
0,1,2,3.5
1,1,3,4.5
2,1,5,3.5
3,1,6,3.5
4,1,11,3.0


### Number of rows and columns in each dataset (1 mark)

In [3]:
shape_summary = pd.DataFrame(
    {"Rows": [movies.shape[0], ratings.shape[0]],
     "Columns": [movies.shape[1], ratings.shape[1]],
     "Column names": [", ".join(movies.columns), ", ".join(ratings.columns)]},
    index=["movies.csv", "ratings.csv"])
display(shape_summary)

print(f"movies.csv  : {movies.shape[0]} rows x {movies.shape[1]} columns")
print(f"ratings.csv : {ratings.shape[0]} rows x {ratings.shape[1]} columns")

,Rows,Columns,Column names
movies.csv,36,3,"movie_id, title, genres"
ratings.csv,440,3,"user_id, movie_id, rating"


movies.csv  : 36 rows x 3 columns
ratings.csv : 440 rows x 3 columns


### Missing-value check (1 mark)

In [4]:
print("Missing values per column - movies.csv")
display(movies.isnull().sum().to_frame("missing_values"))

print("Missing values per column - ratings.csv")
display(ratings.isnull().sum().to_frame("missing_values"))

print(f"Total missing values -> movies.csv: {int(movies.isnull().sum().sum())}, "
      f"ratings.csv: {int(ratings.isnull().sum().sum())}")

# Additional integrity checks, since a clean missing-value report alone says little.
orphans = set(ratings["movie_id"]) - set(movies["movie_id"])
print(f"Duplicate rows            -> movies: {movies.duplicated().sum()}, ratings: {ratings.duplicated().sum()}")
print(f"Duplicate (user, movie)   -> {ratings.duplicated(['user_id', 'movie_id']).sum()}")
print(f"ratings movie_ids missing from the catalogue -> {orphans if orphans else 'none'}")
print(f"Distinct users: {ratings['user_id'].nunique()}  |  Distinct movies rated: {ratings['movie_id'].nunique()}")
print(f"Rating scale observed: {sorted(ratings['rating'].unique())}")

Missing values per column - movies.csv


,missing_values
movie_id,0
title,0
genres,0


Missing values per column - ratings.csv


,missing_values
user_id,0
movie_id,0
rating,0


Total missing values -> movies.csv: 0, ratings.csv: 0
Duplicate rows            -> movies: 0, ratings: 0
Duplicate (user, movie)   -> 0
ratings movie_ids missing from the catalogue -> none
Distinct users: 40  |  Distinct movies rated: 36
Rating scale observed: [np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]


**What I find.** Neither dataset has a single missing value: all three columns of `movies.csv`
(36 rows) and all three of `ratings.csv` (440 rows) report zero nulls. The data is also clean beyond
nulls — there are no duplicate rows, no duplicated `(user_id, movie_id)` pairs, and every `movie_id`
appearing in `ratings.csv` exists in the catalogue, so the two files join cleanly with no orphan
ratings. Ratings are given on a half-point scale from 1.5 to 5.0 by 40 users, and every one of the
36 movies has been rated at least once. No cleaning or imputation is therefore required before
profiling the user.

---
## Task 2 – Profile Your Assigned User (5 marks)

### All movies rated by User 5, with title, genres and rating (2 marks)

In [5]:
# Join the assigned user's ratings onto the catalogue to get titles and genres.
user_ratings = (ratings[ratings["user_id"] == USER_ID]
                .merge(movies, on="movie_id")
                .sort_values("rating", ascending=False)
                .reset_index(drop=True))

print(f"User {USER_ID} has rated {len(user_ratings)} of the {len(movies)} movies in the catalogue "
      f"(mean rating {user_ratings['rating'].mean():.2f}).")
display(user_ratings[["movie_id", "title", "genres", "rating"]])

User 5 has rated 9 of the 36 movies in the catalogue (mean rating 3.72).


,movie_id,title,genres,rating
0,19,The Social Network,Drama|Biography,5.0
1,20,Hidden Figures,Drama|Biography,4.5
2,36,Remember the Titans,Sports|Drama,4.5
3,13,Titanic,Romance|Drama,4.0
4,17,The Pursuit of Happyness,Drama|Biography,4.0
5,33,Ford v Ferrari,Sports|Drama,4.0
6,2,Interstellar,Sci-Fi|Drama,3.0
7,21,Toy Story,Animation|Comedy,2.5
8,8,John Wick,Action|Thriller,2.0


### The user's three highest-rated movies (1 mark)

In [6]:
top3 = user_ratings.nlargest(3, "rating")[["title", "genres", "rating"]].reset_index(drop=True)
top3.index = ["1st", "2nd", "3rd"]
display(top3)

,title,genres,rating
1st,The Social Network,Drama|Biography,5.0
2nd,Hidden Figures,Drama|Biography,4.5
3rd,Remember the Titans,Sports|Drama,4.5


### Evidence for the user's preferred genre(s) (2 marks)

In [7]:
# Explode the pipe-separated genre strings so each (movie, genre) pair is one row,
# then summarise how the user rates each individual genre.
user_genres_long = (user_ratings
                    .assign(genre=user_ratings["genres"].str.split("|"))
                    .explode("genre"))

genre_profile = (user_genres_long
                 .groupby("genre")
                 .agg(movies_rated=("rating", "size"),
                      mean_rating=("rating", "mean"),
                      titles=("title", lambda s: ", ".join(s)))
                 .sort_values(["mean_rating", "movies_rated"], ascending=False)
                 .round(2))

print(f"How User {USER_ID} rates each genre they have actually watched:")
display(genre_profile)

liked_preview = user_ratings[user_ratings["rating"] >= LIKE_THRESHOLD]
print(f"Movies rated >= {LIKE_THRESHOLD}: {len(liked_preview)} of {len(user_ratings)}")
print(f"Of those, the number tagged 'Drama': {liked_preview['genres'].str.contains('Drama').sum()}")

How User 5 rates each genre they have actually watched:


,movies_rated,mean_rating,titles
genre,,,
Biography,3,4.50,"The Social Network, Hidden Figures, The Pursuit of Happyness"
Sports,2,4.25,"Remember the Titans, Ford v Ferrari"
Drama,7,4.14,"The Social Network, Hidden Figures, Remember the Titans, Titanic, The Pursui..."
Romance,1,4.00,Titanic
Sci-Fi,1,3.00,Interstellar
Animation,1,2.50,Toy Story
Comedy,1,2.50,Toy Story
Action,1,2.00,John Wick
Thriller,1,2.00,John Wick


Movies rated >= 4.0: 6 of 9
Of those, the number tagged 'Drama': 6


**The user's apparent preference.** User 5 is clearly a **Drama** viewer, and more specifically a
fan of **Biography** and **Sports** dramas. The evidence is unusually consistent: all six movies they
rated 4.0 or higher carry the `Drama` tag, and their three highest-rated films — *The Social Network*
(5.0), *Hidden Figures* (4.5) and *Remember the Titans* (4.5) — are all either biographical or sports
dramas. Ranked by mean rating, `Biography` averages 4.50 across 3 films, `Sports` 4.25 across 2 and
`Drama` 4.14 across 7, whereas the only `Action`, `Thriller`, `Animation` and `Comedy` titles they
rated sit far below at 2.00–2.50. Because the gap between the drama-family genres and everything else
is both wide and without exception, the profile is a fact-based, character-driven drama viewer rather
than a genre generalist.

---
## Task 3 – Build the Recommendation Logic (7 marks)

A content-based recommender: represent every movie by its genres, measure how similar each unrated
movie is to the movies this user has positively rated, and return the five closest matches.

### Step 1 – Treat ratings of 4.0 or higher as "liked" (1 mark)

In [8]:
liked = user_ratings[user_ratings["rating"] >= LIKE_THRESHOLD].copy()
liked_ids = liked["movie_id"].tolist()

print(f"Movies User {USER_ID} likes (rating >= {LIKE_THRESHOLD}): {len(liked_ids)}")
display(liked[["movie_id", "title", "genres", "rating"]].reset_index(drop=True))

Movies User 5 likes (rating >= 4.0): 6


,movie_id,title,genres,rating
0,19,The Social Network,Drama|Biography,5.0
1,20,Hidden Figures,Drama|Biography,4.5
2,36,Remember the Titans,Sports|Drama,4.5
3,13,Titanic,Romance|Drama,4.0
4,17,The Pursuit of Happyness,Drama|Biography,4.0
5,33,Ford v Ferrari,Sports|Drama,4.0


### Step 2 – Convert genres into numerical features with TF-IDF (2 marks)

In [9]:
# token_pattern=r"[^|]+" keeps each pipe-separated genre as ONE token. The default word
# pattern would split "Sci-Fi" into "sci" + "fi" and invent a genre that does not exist.
vectorizer = TfidfVectorizer(token_pattern=r"[^|]+")
genre_matrix = vectorizer.fit_transform(movies["genres"])

print(f"TF-IDF matrix shape: {genre_matrix.shape}  (movies x genre features)")
print(f"{len(vectorizer.get_feature_names_out())} genre features learned:")
print(", ".join(vectorizer.get_feature_names_out()))

# Show the representation for a few movies so the encoding is visible, not just asserted.
feature_frame = pd.DataFrame(genre_matrix.toarray(),
                             index=movies["title"],
                             columns=vectorizer.get_feature_names_out()).round(2)
print("TF-IDF features for the user's three favourite movies:")
display(feature_frame.loc[top3["title"]].loc[:, lambda d: d.sum() > 0])

TF-IDF matrix shape: (36, 16)  (movies x genre features)
16 genre features learned:
action, adventure, animation, biography, comedy, crime, drama, family, horror, musical, mystery, romance, sci-fi, sports, superhero, thriller
TF-IDF features for the user's three favourite movies:


,biography,drama,sports
title,,,
The Social Network,0.86,0.51,0.00
Hidden Figures,0.86,0.51,0.00
Remember the Titans,0.00,0.54,0.84


### Step 3 – Cosine similarity between movies (2 marks)

In [10]:
similarity = cosine_similarity(genre_matrix)
similarity_df = pd.DataFrame(similarity, index=movies["movie_id"], columns=movies["movie_id"])

print(f"Cosine similarity matrix shape: {similarity_df.shape}")

# Readable view: how similar every liked movie is to the others.
liked_titles = movies.set_index("movie_id").loc[liked_ids, "title"]
liked_block = similarity_df.loc[liked_ids, liked_ids].round(2)
liked_block.index, liked_block.columns = liked_titles, liked_titles
print("Genre similarity among the movies the user likes:")
display(liked_block)

Cosine similarity matrix shape: (36, 36)
Genre similarity among the movies the user likes:


title,The Social Network,Hidden Figures,Remember the Titans,Titanic,The Pursuit of Happyness,Ford v Ferrari
title,,,,,,
The Social Network,1.00,1.00,0.27,0.27,1.00,0.27
Hidden Figures,1.00,1.00,0.27,0.27,1.00,0.27
Remember the Titans,0.27,0.27,1.00,0.29,0.27,1.00
Titanic,0.27,0.27,0.29,1.00,0.27,0.29
The Pursuit of Happyness,1.00,1.00,0.27,0.27,1.00,0.27
Ford v Ferrari,0.27,0.27,1.00,0.29,0.27,1.00


### Step 4 – Score candidates, exclude rated movies, return exactly five (2 marks)

Each candidate is scored by its **mean cosine similarity to the user's six liked movies**, so a film
must resemble the user's taste as a whole rather than just one outlier.

Because genres are coarse (one or two tags per film), several candidates score *identically*. Rather
than accept whatever order `pandas` happens to return, ties are broken by a second, data-driven key:
**genre affinity**, the average rating the user has actually given to that candidate's genres. A
final `movie_id` tiebreak makes the result fully deterministic and reproducible.

In [11]:
# Candidates = every movie the user has NOT already rated. This is the exclusion step.
rated_ids = set(user_ratings["movie_id"])
candidate_ids = [m for m in movies["movie_id"] if m not in rated_ids]

print(f"Catalogue: {len(movies)} movies | already rated by User {USER_ID}: {len(rated_ids)} "
      f"| candidates remaining: {len(candidate_ids)}")

# Primary score: mean cosine similarity to the liked set.
mean_similarity = similarity_df.loc[candidate_ids, liked_ids].mean(axis=1)

# Tiebreak key: mean rating the user gave to the genres this candidate carries.
genre_means = user_genres_long.groupby("genre")["rating"].mean()
user_mean = user_ratings["rating"].mean()

def genre_affinity(genre_string):
    seen = [genre_means[g] for g in genre_string.split("|") if g in genre_means]
    return float(np.mean(seen)) if seen else user_mean   # neutral if genre unseen by this user

candidates = (movies[movies["movie_id"].isin(candidate_ids)]
              .assign(mean_similarity=lambda d: d["movie_id"].map(mean_similarity).round(3),
                      genre_affinity=lambda d: d["genres"].map(genre_affinity).round(2))
              .sort_values(["mean_similarity", "genre_affinity", "movie_id"],
                           ascending=[False, False, True])
              .reset_index(drop=True))

print("All 27 candidates, ranked (note the three-way tie at 0.303 resolved by genre affinity):")
display(candidates[["movie_id", "title", "genres", "mean_similarity", "genre_affinity"]].head(10))

recommendations = candidates.head(N_RECOMMENDATIONS).reset_index(drop=True)

print(f"The {N_RECOMMENDATIONS} recommended movie titles for User {USER_ID}:")
for i, title in enumerate(recommendations["title"], start=1):
    print(f"  {i}. {title}")

Catalogue: 36 movies | already rated by User 5: 9 | candidates remaining: 27
All 27 candidates, ranked (note the three-way tie at 0.303 resolved by genre affinity):


,movie_id,title,genres,mean_similarity,genre_affinity
0,18,The Shawshank Redemption,Drama,0.522,4.14
1,34,Creed,Sports|Drama,0.517,4.20
2,35,Moneyball,Sports|Drama,0.517,4.20
3,14,The Notebook,Romance|Drama,0.398,4.07
4,4,Arrival,Sci-Fi|Drama,0.303,3.57
5,6,Blade Runner 2049,Sci-Fi|Drama,0.303,3.57
6,30,The Grand Budapest Hotel,Comedy|Drama,0.303,3.32
7,16,Crazy Rich Asians,Romance|Comedy,0.105,3.25
8,15,La La Land,Romance|Musical,0.086,4.00
9,27,The Conjuring,Horror,0.000,3.72


The 5 recommended movie titles for User 5:
  1. The Shawshank Redemption
  2. Creed
  3. Moneyball
  4. The Notebook
  5. Arrival


### Compliance check

The two hard constraints of the brief, verified rather than assumed.

In [12]:
assert len(recommendations) == N_RECOMMENDATIONS, "must return exactly five recommendations"
overlap = set(recommendations["movie_id"]) & rated_ids
assert not overlap, f"recommended an already-rated movie: {overlap}"

print(f"PASS - exactly {len(recommendations)} recommendations returned.")
print(f"PASS - none of the recommendations appear in User {USER_ID}'s rated movies.")
print(f"        recommended movie_ids = {sorted(recommendations['movie_id'])}")
print(f"        already-rated movie_ids = {sorted(rated_ids)}")

PASS - exactly 5 recommendations returned.
PASS - none of the recommendations appear in User 5's rated movies.
        recommended movie_ids = [4, 14, 18, 34, 35]
        already-rated movie_ids = [2, 8, 13, 17, 19, 20, 21, 33, 36]


---
## Task 4 – Explain the Recommendations (3 marks)

In [13]:
movie_lookup = movies.set_index("movie_id")
liked_lookup = liked.set_index("movie_id")

def build_reason(movie_id):
    """Explain a recommendation via the liked movie it most closely resembles."""
    sims = similarity_df.loc[movie_id, liked_ids]
    best_id = sims.idxmax()
    best_title = liked_lookup.loc[best_id, "title"]
    best_rating = liked_lookup.loc[best_id, "rating"]
    shared = (set(movie_lookup.loc[movie_id, "genres"].split("|"))
              & set(liked_lookup.loc[best_id, "genres"].split("|")))
    shared_text = " + ".join(sorted(shared))
    return (f"Shares {shared_text} with \"{best_title}\" (user rated {best_rating}) "
            f"- similarity {sims.max():.2f}; mean similarity to all liked movies "
            f"{mean_similarity[movie_id]:.2f}.")

results_table = pd.DataFrame({
    "Rank": range(1, len(recommendations) + 1),
    "Recommended Movie": recommendations["title"],
    "Genres": recommendations["genres"],
    "Reason": [build_reason(m) for m in recommendations["movie_id"]],
})

print(f"Final recommendations for User {USER_ID} - Justice Chawanda (670444)")
display(results_table.set_index("Rank"))

Final recommendations for User 5 - Justice Chawanda (670444)


,Recommended Movie,Genres,Reason
Rank,,,
1,The Shawshank Redemption,Drama,"Shares Drama with ""Remember the Titans"" (user rated 4.5) - similarity 0.54; ..."
2,Creed,Sports|Drama,"Shares Drama + Sports with ""Remember the Titans"" (user rated 4.5) - similari..."
3,Moneyball,Sports|Drama,"Shares Drama + Sports with ""Remember the Titans"" (user rated 4.5) - similari..."
4,The Notebook,Romance|Drama,"Shares Drama + Romance with ""Titanic"" (user rated 4.0) - similarity 1.00; me..."
5,Arrival,Sci-Fi|Drama,"Shares Drama with ""Remember the Titans"" (user rated 4.5) - similarity 0.31; ..."


**Interpretation.** The five recommendations track User 5's observed profile closely. *Creed* and
*Moneyball* are exact genre matches (`Sports|Drama`, similarity 1.00) for *Ford v Ferrari* and
*Remember the Titans*, two films the user rated 4.0 and 4.5 — the sports-drama signal in their
history is reproduced directly. *The Shawshank Redemption* ranks first overall because, as pure
`Drama`, it sits closest to the single genre common to **all six** of the user's liked movies.
*The Notebook* is a perfect `Romance|Drama` match for *Titanic* (4.0), and *Arrival* wins the fifth slot
on the tiebreak described above. It is the most speculative of the five: its `Drama` half is well
supported, but its `Sci-Fi` half rests only on *Interstellar*, the catalogue's other `Sci-Fi|Drama`
film, which the user rated just 3.0 — below the "liked" threshold. That makes it a sensible, clearly
caveated place for the list to end.

---
## Task 5 – Critical Thinking (2 marks)

### One limitation of this recommender (1 mark)

The recommender knows a movie only by its genre tags, and those tags are far too coarse to separate
films that are genuinely different. Each movie carries just one or two of 16 labels, so similarity is
blunt and heavily tied: *Creed* and *Moneyball* are mathematically **indistinguishable** here (both
score exactly 0.517), and three candidates tied at 0.303 for the fifth slot, which is why an
additional tiebreak rule was needed at all. The approach is also purely content-based, so it can only
return more of what the user already watches — all five recommendations are dramas, and nothing can
surface a film the user would love for reasons genres do not capture, such as its director, cast,
era or tone. The binary 4.0 cutoff compounds this by discarding information: *The Social Network*
(5.0) and *Titanic* (4.0) carry identical weight in the liked set even though the user rated one a
full point higher.

### One realistic improvement (1 mark)

The most practical improvement is to make the system **hybrid** by adding a collaborative-filtering
signal from the 440 ratings already in `ratings.csv`. Computing user–user similarity over the 40
users would let the recommender find users whose rating patterns resemble User 5's and surface movies
those neighbours rated highly — recovering exactly the cross-genre discoveries that genre similarity
cannot reach, and breaking the ties above with real behavioural evidence instead of an arbitrary
rule. A cheaper refinement in the same direction is to **weight each liked movie by its rating** when
averaging similarities, so a 5.0 film pulls the recommendations harder than a 4.0 one, and to enrich
the feature set with director, cast, release year and plot keywords so that "Drama" stops being a
single undifferentiated bucket.